In [1]:
# imports and settings

import os
import time
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch, get_window
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import utils as ut
from reference_detector.ship_noise_pkl_model import run_model as run_reference_model

# do not show warnings
warnings.filterwarnings("ignore")

# autoreload for development
%load_ext autoreload
%autoreload 2

print("Imports complete.")

Imports complete.
Settings: height=800, width=1400, font_size=16
Imports complete.


In [2]:
folder_path = '../data/Garda_06022026/2_deep_water/Petrol/'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

# data = []
# for file in file_list:
#     print(f"Loading file: {file}")
#     _data, fs = librosa.load(os.path.join(folder_path, file))
#     data.append(_data)
# data = np.concatenate(data)
# print(f"data final shape: {data.shape}")

data, fs = librosa.load(os.path.join(folder_path, file_list[3]))

In [8]:
nperseg = 8000
overlap = 0.5
window = 'hanning'
dc = 20
crop_freq = 8000
norm_size = 5
s2g_quantization_levels = 3
s2g_mode = 'laplacian'
welch_threshold = 0.034
s2g_threshold = 0.3
reference_model_threshold = 0.5

welch_detector = ut.WelchDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
s2g_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=s2g_quantization_levels)


In [14]:
# sanity

_data = data[1*60*fs:2*60*fs]
F, T, Sxx, phasogram = ut.calc_spectrogram(_data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)
pxx, _ = welch_detector.get_feature_vector(_data)
K, _ = s2g_detector.get_feature_vector(_data)

# plot Spectrogram
fig = make_subplots(rows=1, cols=3, column_widths=[0.8, 0.1, 0.1], shared_yaxes=True)
fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=pxx, y=F, mode='lines', name='Welch'), row=1, col=2)
fig.add_trace(go.Scatter(x=K, y=F, mode='lines', name='S2G'), row=1, col=3)
fig.update_layout(height=800, width=1200, title_text="Spectrogram and Detection Metrics for All Recordings")
fig.show(renderer="browser")
# 
fig.write_html("../results/nb83_analysis_190326_2nd_minute.html")

In [10]:
# plot Spectrogram
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

welch_detections = []
welch_max_scores = []
welch_detection_times = []

s2g_detections = []
s2g_max_scores = []
s2g_detection_times = []

reference_model_detections = []

step_size = 30 * fs  # 30 seconds overlap
for si in np.arange(start=0, stop=len(data), step=step_size):
    _data = data[si:si+2*step_size]
    _time = (si + step_size) // fs  # time corresponding to the center of the window
    
    w_is_d , wd, (pxx, F) = welch_detector.detect(_data, threshold=welch_threshold)
    welch_max_scores.append(np.max(pxx))

    if w_is_d:
        for _wd in wd:
            welch_detection_times.append(_time)
            welch_detections.append(_wd)

    s2g_is_d, s2gd, (K, F) = s2g_detector.detect(_data, threshold=s2g_threshold)
    s2g_max_scores.append(np.max(K))

    if s2g_is_d:
        for _s2gd in s2gd:
            s2g_detection_times.append(_time)
            s2g_detections.append(_s2gd)

    reference_detection = run_reference_model(reference_model_threshold, torch.from_numpy(_data.astype(np.float32)))
    reference_model_detections.append(reference_detection['probability_ship'])

detectors_taxis = np.arange(start=step_size, stop=len(data), step=step_size) / fs

In [16]:
fig = make_subplots(rows=4, 
                    cols=1, 
                    row_heights=[0.7, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    row_titles=["Spectrogram", "Lab", "Welch-CFAR", "S2G"])

fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=welch_detection_times, y=F[welch_detections], mode='markers', name='Welch Scores', marker_color='blue', marker_symbol='circle', marker_size=8), row=1, col=1)
fig.add_trace(go.Scatter(x=s2g_detection_times, y=F[s2g_detections], mode='markers', name='S2G Scores', marker_color='purple', marker_symbol='x', marker_size=6), row=1, col=1)

fig.add_trace(go.Bar(x=detectors_taxis, y=reference_model_detections, name='Reference Model Detections', marker_color='green'), row=2, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=reference_model_threshold*np.ones_like(detectors_taxis), name='Reference Model Threshold', marker_color='red', line=dict(dash='dash')), row=2, col=1)

fig.add_trace(go.Bar(x=detectors_taxis, y=welch_max_scores, name='Welch Detections', marker_color='blue'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=welch_threshold*np.ones_like(detectors_taxis), name='Welch Threshold', marker_color='red', line=dict(dash='dash')), row=3, col=1)

fig.add_trace(go.Bar(x=detectors_taxis, y=s2g_max_scores, name='S2G Detections', marker_color='purple'), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=s2g_threshold*np.ones_like(detectors_taxis), name='S2G Threshold', marker_color='red', line=dict(dash='dash')), row=4, col=1)

fig.update_layout(height=800, width=1200, title_text=f"Spectrogram and Detection Metrics for recording: {file_list[3]}")
fig.show(renderer="browser")

In [12]:
fig.write_html("../results/nb83_analysis_190326.html")